# 03 – Unsupervised Clustering

## Syfte

Denna notebook utforskar ansiktsembeddings (512-dimensionella ArcFace-vektorer)
genom klustring, för att upptäcka strukturer i datasetet utan att luta oss mot
kända etiketter. Målet är dubbelt:

1. **Demonstrera unsupervised learning** enligt kursens krav — undersöka om
   embedding-rymden naturligt separerar ansikten efter demografiska drag
   (t.ex. ålder, kön) eller andra mönster, utan att modellen tränas mot
   dessa etiketter.
2. **Generera underlag för senare steg** — de embeddings som extraheras här
   sparas till disk och återanvänds i notebook 04
   (`04_supervised_classification.ipynb`) för den auktoriserad/ej
   auktoriserad-klassificeraren samt ålder/kön-modellen.

## Arbetsgång

1. Ladda den filtrerade metadatan (resultat av `preprocessing.py`-flaggorna).
2. Extrahera embeddings för samtliga kvarvarande bilder, i chunkar om 750
   bilder åt gången, med stöd för att återuppta en avbruten körning
   (se `extract_embeddings_chunked()` i `src/embeddings.py`).
3. Slå ihop chunkarna till en samlad `embeddings.npy` + `embeddings_index.csv`.
4. Koppla embeddings tillbaka till metadata (ålder, kön) via radindex.
5. Kluster embeddings med K-means och/eller UMAP, och tolka resultatet.

**Notera:** Steg 2 (embedding-extraktion över hela datasetet) är
beräkningstungt (~2,5–3 timmar på CPU) och körs som ett bakgrundsjobb.
Koden verifieras först på ett litet urval innan den körs i sin helhet.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import umap

# Add project root (parent of notebooks/) to sys.path so `src` is importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.embeddings import extract_embeddings_chunked  # type: ignore

## Ladda flaggad metadata och filtrera på detekterat ansikte

Vi laddar den redan flaggade metadatan (`wiki_metadata_flagged.parquet`,
sparad i steg 02) och tillämpar `filter_valid_faces()` för att behålla
endast rader där ett ansikte faktiskt detekterades i originalbilden.
Detta är en förutsättning för embedding-extraktion — en bild utan
detekterat ansikte kan inte ge en meningsfull embedding.

Övriga flaggor (`gender_missing`, `age_implausible`) filtreras **inte**
bort här, eftersom de inte påverkar möjligheten att extrahera en
embedding. De hanteras separat, senare, där de faktiskt blir relevanta
(t.ex. vid tolkning av kluster mot demografi, eller vid träning av
ålder/kön-modellen i notebook 04).

In [2]:
from src.preprocessing import filter_valid_faces

df = pd.read_parquet("../data/processed/wiki_metadata_flagged.parquet")
print(f"Inläst: {len(df)} rader")

df_valid = filter_valid_faces(df)
print(f"Efter filter_valid_faces(): {len(df_valid)} rader "
      f"({len(df) - len(df_valid)} rader bortfiltrerade)")

Inläst: 62328 rader
Efter filter_valid_faces(): 44312 rader (18016 rader bortfiltrerade)


### Resultat

**44 312 av 62 328 rader** (71,09%) har ett detekterat ansikte och går
vidare till embedding-extraktion. De 18 016 bortfiltrerade raderna
(28,91%) saknar `face_location`-data och kan därför inte beskäras eller
representeras som en meningsfull embedding — detta matchar exakt den
andel som identifierades i steg 01/02.